# petri clash build walkthrough

this notebook is a full walkthrough of how the project is built, why each file exists, what the code is doing, and what i was thinking while putting it together.

this is not a polished research note. it is closer to a very long engineering brain dump. the point is to explain the real implementation as it exists in this repo, including the mistakes, the fixes, and the tradeoffs.

when i build something like this, i usually split my thinking into four layers:

1. what is the smallest version that is actually the same idea.
2. what parts need to be faithful to the original nca paper.
3. what parts can be simplified so the whole thing stays understandable.
4. what parts are likely to break once two models start sharing space.

petri clash ended up following exactly that path.

## repo shape

before touching details, this is the mental map of the repo.

- `nca.py` is the core learned cellular automaton rule.
- `train.py` trains one nca against one target image.
- `clash.py` loads two trained ncas, places them on one board, and handles the fight.
- `targets/` contains the little icon images.
- `weights/` contains saved checkpoints.

that split is deliberate. i did not want the game loop to know anything about optimization internals, and i did not want the trainer to know anything about pygame.

## the big design idea

the whole project only works if one fact stays true:

an nca is not drawing pixels directly in the way a normal image model would. it is learning a local update law.

that means each cell only gets a tiny local view, updates its own 16-channel state, and then repeats this over and over. so the model is really learning a dynamical system, not a one-shot renderer.

that is why the project feels alive when it works. the same learned rule can:

- grow from a seed
- repair damage
- continue evolving after partial destruction
- compete at a frontier

that also means failure modes are dynamical failure modes. if the update rule gets pushed outside what it saw during training, it does not fail gracefully. it starts inventing weird junk, drifting, leaving sparks behind, and feeding on those mistakes.

## file 1: `nca.py`

this file is the heart of everything.

when i started building it, i tried to keep the model as close as possible to the 2020 distill setup while still staying small enough to understand at a glance.

the choices here are:

- 16 channels per cell
- rgba in the first 4 channels
- 12 hidden channels after that
- depthwise perception using identity, sobel-x, and sobel-y
- a tiny per-cell mlp implemented as two 1x1 convolutions
- stochastic updates with a fire rate of 0.5
- alive masking based on alpha

that is almost the whole nca idea in one page of code.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F


class NCA(nn.Module):
    def __init__(self, channels=16, hidden_size=128, fire_rate=0.5):
        super().__init__()
        self.channels = channels
        self.hidden_size = hidden_size
        self.fire_rate = fire_rate

        sobel_x = torch.tensor(
            [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]
        ) / 8.0
        sobel_y = sobel_x.t()
        identity = torch.tensor(
            [[0.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 0.0]]
        )

        filters = torch.stack([identity, sobel_x, sobel_y], dim=0)[:, None]
        filters = filters.repeat(channels, 1, 1, 1)
        self.register_buffer("perception_filters", filters)

        self.fc0 = nn.Conv2d(channels * 3, hidden_size, 1)
        self.fc1 = nn.Conv2d(hidden_size, channels, 1)
        nn.init.zeros_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)

    def perceive(self, x):
        return F.conv2d(x, self.perception_filters, padding=1, groups=self.channels)

    def alive_mask(self, x):
        alpha = x[:, 3:4]
        return F.max_pool2d(alpha, 3, stride=1, padding=1) > 0.1

    def step(self, x, fire_rate=None):
        if fire_rate is None:
            fire_rate = self.fire_rate

        pre_life = self.alive_mask(x)
        y = self.perceive(x)
        dx = self.fc1(F.relu(self.fc0(y)))

        if fire_rate < 1.0:
            # this keeps updates patchy so the thing has to work asynchronously
            mask = (torch.rand_like(x[:, :1]) <= fire_rate).float()
            dx = dx * mask

        x = x + dx
        post_life = self.alive_mask(x)
        x = x * (pre_life | post_life).float()
        return x

    def forward(self, x, steps=1, fire_rate=None):
        for _ in range(steps):
            x = self.step(x, fire_rate=fire_rate)
        return x


def make_seed(batch_size, channels=16, height=48, width=None, xs=None, ys=None, device=None):
    if width is None:
        width = height

    state = torch.zeros(batch_size, channels, height, width, device=device)
    if xs is None:
        xs = torch.full((batch_size,), width // 2, dtype=torch.long, device=device)
    else:
        xs = torch.as_tensor(xs, dtype=torch.long, device=device)
    if ys is None:
        ys = torch.full((batch_size,), height // 2, dtype=torch.long, device=device)
    else:
        ys = torch.as_tensor(ys, dtype=torch.long, device=device)

    batch_ids = torch.arange(batch_size, device=device)

    # one alpha pulse and one hidden spark is enough to get things moving
    state[batch_ids, 3, ys, xs] = 1.0
    state[batch_ids, 4, ys, xs] = 1.0
    return state


def pick_device():
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


## why the state is 16 channels

four visible channels are not enough.

if i only let each cell keep rgba, the model has almost no working memory. it can paint, but it cannot carry enough latent structure to build shape, repair structure, or negotiate boundaries in an interesting way.

the hidden channels are the model's private scratch space. they are where the internal wavefront, orientation cues, and whatever other latent bookkeeping the model invents can live.

that is why the hidden state is not optional decoration. it is what makes the organism behavior possible.

## why sobel filters show up here

the perception stage is fixed, not learned.

that sounds restrictive at first, but it is actually helpful. each channel gets:

- its own center value through the identity filter
- horizontal differences through sobel-x
- vertical differences through sobel-y

so each cell can cheaply sense both "what am i" and "how is the local pattern changing around me".

using fixed filters keeps the parameter count tiny and makes the system closer to the distill recipe. it also means the learning burden stays in the update rule rather than in a larger perception stack.

In [ ]:
import torch
from nca import NCA

model = NCA()
print('perception filter shape:', tuple(model.perception_filters.shape))
print('fc0 weight shape:', tuple(model.fc0.weight.shape))
print('fc1 weight shape:', tuple(model.fc1.weight.shape))
print('parameter count:', sum(p.numel() for p in model.parameters()))

## why the last layer starts at zero

this is one of those tiny details that matters more than it looks.

if the last layer starts random, then the very first rollout from the seed can be chaotic. the whole grid can explode before learning finds any stable attractor.

if the last layer starts at zero, the model initially behaves like "do almost nothing". then training gradually learns useful residual updates.

that gives a much calmer optimization path.

## the alive mask

this is the line that decides whether dead regions really stay dead.

alpha is treated as the cell's visible aliveness signal. if alpha is weak in a local 3x3 area, the whole cell state gets zeroed out.

there are two reasons for this:

- it prevents invisible hidden junk from surviving forever after visible structure dies
- it keeps the growth front compact instead of letting latent noise linger in dead space

this becomes even more important once damage craters and collisions are involved.

In [ ]:
from nca import make_seed

state = make_seed(1, height=48, width=48, device='cpu')
print('state shape:', tuple(state.shape))
print('seed alpha sum:', float(state[:, 3].sum()))
print('seed hidden channel 4 sum:', float(state[:, 4].sum()))

## file 2: `train.py`

this file exists because a clash game is useless if the two organisms are not real organisms first.

so the training loop had to do a few specific jobs:

- load a target image and put it on the same kind of grid the nca uses
- keep a pool of partially grown states instead of starting from scratch every time
- occasionally reinsert a fresh seed so the model still learns how to grow from nothing
- save checkpoints with enough metadata that the game can decide whether a checkpoint is usable
- later on, inject random damage during training because the clash world contains holes and disruptions

this file started simple and then got more serious once the first clash runs showed how easily ncas drift outside their comfort zone.

In [ ]:
import argparse
import random
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from nca import NCA, make_seed, pick_device


def load_target(path, target_size=40, grid_size=48, device="cpu"):
    image = Image.open(path).convert("RGBA")
    image = image.resize((target_size, target_size), Image.Resampling.LANCZOS)

    canvas = Image.new("RGBA", (grid_size, grid_size), (0, 0, 0, 0))
    offset = ((grid_size - target_size) // 2, (grid_size - target_size) // 2)
    canvas.paste(image, offset, image)

    array = np.asarray(canvas, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0)
    return tensor.to(device)


def save_checkpoint(model, out_path, args, target_path, train_steps):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    blob = {
        "state_dict": model.state_dict(),
        "target_name": target_path.stem,
        "target_path": str(target_path),
        "train_steps": train_steps,
        "grid_size": args.grid_size,
        "target_size": args.target_size,
        "channels": model.channels,
        "hidden_size": model.hidden_size,
        "fire_rate": model.fire_rate,
    }
    torch.save(blob, out_path)


def damage_batch(batch, damage_prob=0.5):
    if damage_prob <= 0:
        return batch

    size = batch.shape[-1]
    yy = torch.arange(size, device=batch.device).view(1, 1, size, 1)
    xx = torch.arange(size, device=batch.device).view(1, 1, 1, size)

    for index in range(1, batch.shape[0]):
        if random.random() > damage_prob:
            continue
        radius = random.randint(max(2, size // 8), max(3, size // 4))
        cx = random.randint(radius, size - radius - 1)
        cy = random.randint(radius, size - radius - 1)
        crater = ((xx - cx).pow(2) + (yy - cy).pow(2)) <= radius * radius
        batch[index] = torch.where(
            crater.expand_as(batch[index : index + 1]),
            torch.zeros_like(batch[index : index + 1]),
            batch[index : index + 1],
        )[0]
    return batch


def train_target(
    target_path,
    out_path=None,
    steps=3000,
    pool_size=1024,
    batch_size=8,
    grid_size=48,
    target_size=40,
    min_rollout=64,
    max_rollout=96,
    lr=2e-3,
    save_every=250,
    seed=0,
    device=None,
    damage_prob=0.5,
    resume=True,
):
    if device is None:
        device = pick_device()

    target_path = Path(target_path)
    if not target_path.exists():
        raise SystemExit(f"missing target: {target_path}")

    if out_path is None:
        out_path = Path("weights") / f"{target_path.stem}.pt"
    else:
        out_path = Path(out_path)

    if seed:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

    args = argparse.Namespace(
        grid_size=grid_size,
        target_size=target_size,
    )

    target = load_target(
        target_path,
        target_size=target_size,
        grid_size=grid_size,
        device=device,
    )

    model = NCA().to(device)
    start_step = 0
    if resume and out_path.exists():
        blob = torch.load(out_path, map_location=device)
        if blob.get("target_name") == target_path.stem:
            model.load_state_dict(blob["state_dict"])
            start_step = int(blob.get("train_steps", 0) or 0)
            print(f"resuming {target_path.name} from step {start_step}")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    pool = make_seed(
        pool_size,
        height=grid_size,
        width=grid_size,
        device=device,
    )
    fresh_seed = make_seed(
        1,
        height=grid_size,
        width=grid_size,
        device=device,
    )[0]

    save_every = max(1, save_every)

    if start_step >= steps:
        print(f"{target_path.name} already has {start_step} steps")
        return out_path

    print(f"training {target_path.name} on {device}")
    start = time.time()
    ema = None

    for step in range(start_step + 1, steps + 1):
        batch_ids = torch.randint(pool_size, (batch_size,), device=device)
        batch = pool[batch_ids].clone()

        # one fresh seed in the batch keeps the organism honest
        batch[0] = fresh_seed
        batch = damage_batch(batch, damage_prob=damage_prob)

        rollout = random.randint(min_rollout, max_rollout)
        batch = model(batch, steps=rollout)
        losses = (batch[:, :4] - target).pow(2).mean(dim=(1, 2, 3))
        loss = losses.mean()

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        batch = batch.detach()
        pool[batch_ids] = batch
        pool[batch_ids[losses.argmax()]] = fresh_seed

        value = float(loss.detach())
        ema = value if ema is None else ema * 0.98 + value * 0.02

        if step % 50 == 0 or step == 1 or step == steps:
            elapsed = max(time.time() - start, 1e-6)
            speed = (step - start_step) / elapsed
            print(
                f"step {step:4d}/{steps} loss {value:.5f} ema {ema:.5f} "
                f"rollout {rollout:2d} speed {speed:.2f} it/s"
            )

        if step % save_every == 0 or step == steps:
            save_checkpoint(model, out_path, args, target_path, step)

    print(f"saved {out_path}")
    return out_path


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--target", required=True)
    parser.add_argument("--out")
    parser.add_argument("--steps", type=int, default=3000)
    parser.add_argument("--pool-size", type=int, default=1024)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--grid-size", type=int, default=48)
    parser.add_argument("--target-size", type=int, default=40)
    parser.add_argument("--min-rollout", type=int, default=64)
    parser.add_argument("--max-rollout", type=int, default=96)
    parser.add_argument("--lr", type=float, default=2e-3)
    parser.add_argument("--save-every", type=int, default=250)
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--device", default=pick_device())
    parser.add_argument("--damage-prob", type=float, default=0.5)
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--no-resume", action="store_true")
    args = parser.parse_args()

    train_target(
        args.target,
        out_path=args.out,
        steps=args.steps,
        pool_size=args.pool_size,
        batch_size=args.batch_size,
        grid_size=args.grid_size,
        target_size=args.target_size,
        min_rollout=args.min_rollout,
        max_rollout=args.max_rollout,
        lr=args.lr,
        save_every=args.save_every,
        seed=args.seed,
        device=args.device,
        damage_prob=args.damage_prob,
        resume=args.resume or not args.no_resume,
    )


if __name__ == "__main__":
    main()


## target loading

one detail i cared about here was matching the training canvas and the runtime canvas.

this is why targets are:

- resized to 40x40
- pasted onto a 48x48 transparent canvas

that gives the organism room to grow inward and outward around the shape, while still keeping the target compact. if the target touched the edges directly, the learned dynamics would be much touchier.

In [ ]:
from train import load_target

target = load_target('targets/01_heart.png', device='cpu')
print('target tensor shape:', tuple(target.shape))
print('rgba min/max:', float(target.min()), float(target.max()))
print('nonzero alpha cells:', int((target[:, 3] > 0).sum()))

## why the pool matters

without a pool, training would mostly teach one thing: how to grow from a clean seed.

that is not enough.

i wanted the model to also learn how to continue from awkward partial states. the pool helps with that because every batch is sampled from a bag of already-grown organisms at different stages.

so the model is constantly asked to fix, continue, and stabilize states that are not identical.

that makes the final dynamics much more robust.

## why there is always one fresh seed in each batch

this is a small but important bias.

if i only trained from pooled states, the model could slowly forget how to bootstrap itself from a single live pixel. it could become good at "keep going" but bad at "start".

forcing one fresh seed into every batch keeps that skill alive.

## why i added `damage_batch`

this came later.

at first i only trained plain growth. then the moment i started thinking about clash and mouse-made craters, it was obvious that the runtime world would include missing chunks and weird wounds.

so i added damage during training.

for some fraction of the batch, i punch out a circular hole before the rollout. this forces the nca to solve not just "grow toward target" but also "repair a broken organism and still end up looking like the target".

that was one of the more practical changes in the whole project.

## why checkpoint metadata matters

this one was learned the hard way.

early on, i ran tiny smoke-training jobs just to verify that the code executed. those low-step weights were good enough for a unit test, but terrible as real organisms.

then the game loaded them because a `.pt` file existed, and the result was basically white blobs and nonsense.

so i changed the checkpoints to save `train_steps`. that let the game say:

- if this checkpoint is undertrained, do not trust it
- if bootstrap training is requested, refresh weak weights instead of blindly loading them

this is the kind of detail that feels boring until it saves you from staring at broken output for an hour.

## file 3: `clash.py`

this file is where the project stopped being a clean training exercise and became an engineering problem.

a single nca growing alone is conceptually simple. two ncas sharing a board is where all the strange edge cases start.

i ended up rewriting the clash logic more than once, because the first obvious version was wrong in a deep way.

In [ ]:
import argparse
import os
import random
from pathlib import Path

import numpy as np
import pygame
import torch
import torch.nn.functional as F

from nca import NCA, make_seed, pick_device
from train import train_target


def list_targets():
    return sorted(Path("targets").glob("*.png"))


def load_model_blob(blob, device):
    model = NCA(
        channels=blob.get("channels", 16),
        hidden_size=blob.get("hidden_size", 128),
        fire_rate=blob.get("fire_rate", 0.5),
    ).to(device)
    model.load_state_dict(blob["state_dict"])
    model.eval()
    return model


def load_model(weight_path, device):
    blob = torch.load(weight_path, map_location=device)
    return load_model_blob(blob, device)


def ensure_model(target_path, device, bootstrap_steps):
    weight_path = Path("weights") / f"{target_path.stem}.pt"
    if weight_path.exists():
        blob = torch.load(weight_path, map_location="cpu")
        trained_steps = int(blob.get("train_steps", 0) or 0)
        if trained_steps >= bootstrap_steps or bootstrap_steps <= 0:
            print(f"loaded {weight_path.name} ({trained_steps} steps)")
            return load_model_blob(blob, device)
        print(
            f"refreshing {weight_path.name} because it only has "
            f"{trained_steps} steps"
        )

    if bootstrap_steps > 0:
        print(f"bootstrapping {target_path.name} for {bootstrap_steps} steps")
        train_target(
            target_path,
            out_path=weight_path,
            steps=bootstrap_steps,
            pool_size=256,
            batch_size=4,
            min_rollout=32,
            max_rollout=48,
            save_every=bootstrap_steps,
            device=device,
        )
        return load_model(weight_path, device)

    print(f"missing {weight_path.name}, using an untrained rule")
    model = NCA().to(device)
    model.eval()
    return model


def clamp(value, low, high):
    return max(low, min(high, value))


def random_seed_positions(size):
    span = max(2, size // 10)
    y = clamp(size // 2 + random.randint(-span, span), 2, size - 3)
    ax = clamp(size // 4 + random.randint(-span, span), 2, size - 3)
    bx = clamp((size * 3) // 4 + random.randint(-span, span), 2, size - 3)

    if abs(ax - bx) < size // 4:
        bx = clamp(ax + size // 2, 2, size - 3)

    return ax, y, bx, y


def reset_world(size, device):
    ax, ay, bx, by = random_seed_positions(size)
    state_a = torch.zeros(1, 16, size, size, device=device)
    state_b = torch.zeros(1, 16, size, size, device=device)
    owner = torch.zeros(1, 1, size, size, dtype=torch.long, device=device)

    state_a = make_seed(1, height=size, width=size, xs=[ax], ys=[ay], device=device)
    state_b = make_seed(1, height=size, width=size, xs=[bx], ys=[by], device=device)

    owner[0, 0, ay, ax] = 1
    owner[0, 0, by, bx] = 2
    return state_a, state_b, owner


def crater(state_a, state_b, owner, gx, gy, radius):
    h, w = state_a.shape[-2:]
    yy = torch.arange(h, device=state_a.device).view(1, 1, h, 1)
    xx = torch.arange(w, device=state_a.device).view(1, 1, 1, w)
    mask = ((xx - gx).pow(2) + (yy - gy).pow(2)) <= radius * radius

    state_a = torch.where(mask.expand_as(state_a), torch.zeros_like(state_a), state_a)
    state_b = torch.where(mask.expand_as(state_b), torch.zeros_like(state_b), state_b)
    owner = torch.where(mask, torch.zeros_like(owner), owner)
    return state_a, state_b, owner


def clash_step(state_a, state_b, owner, model_a, model_b):
    with torch.inference_mode():
        proposed_a = model_a(state_a, steps=1)
        proposed_b = model_b(state_b, steps=1)

        alpha_a = proposed_a[:, 3:4].clamp(0.0, 1.0)
        alpha_b = proposed_b[:, 3:4].clamp(0.0, 1.0)

        owned_a = owner == 1
        owned_b = owner == 2
        near_a = F.max_pool2d(owned_a.float(), 3, stride=1, padding=1) > 0
        near_b = F.max_pool2d(owned_b.float(), 3, stride=1, padding=1) > 0

        # each model keeps its own hidden state now, otherwise they poison each other
        claim_a = (alpha_a > 0.05) & (owned_a | near_a)
        claim_b = (alpha_b > 0.05) & (owned_b | near_b)

        next_owner = owner.clone()
        only_a = claim_a & ~claim_b
        only_b = claim_b & ~claim_a
        both = claim_a & claim_b

        next_owner = torch.where(only_a, torch.ones_like(next_owner), next_owner)
        next_owner = torch.where(only_b, torch.full_like(next_owner, 2), next_owner)

        a_wins = alpha_a >= alpha_b
        winner_owner = torch.where(a_wins, torch.ones_like(owner), torch.full_like(owner, 2))
        next_owner = torch.where(both, winner_owner, next_owner)

        next_state_a = torch.where((next_owner == 1).expand_as(proposed_a), proposed_a, torch.zeros_like(proposed_a))
        next_state_b = torch.where((next_owner == 2).expand_as(proposed_b), proposed_b, torch.zeros_like(proposed_b))

        # tiny sparks are what turn into the ugly late-stage creep, so kill them early
        support_a = F.avg_pool2d((next_owner == 1).float(), 3, stride=1, padding=1)
        support_b = F.avg_pool2d((next_owner == 2).float(), 3, stride=1, padding=1)
        stable_a = (next_owner == 1) & (support_a >= (2.0 / 9.0))
        stable_b = (next_owner == 2) & (support_b >= (2.0 / 9.0))

        next_state_a = torch.where(stable_a.expand_as(next_state_a), next_state_a, torch.zeros_like(next_state_a))
        next_state_b = torch.where(stable_b.expand_as(next_state_b), next_state_b, torch.zeros_like(next_state_b))
        next_owner = torch.where((next_owner == 1) & ~stable_a, torch.zeros_like(next_owner), next_owner)
        next_owner = torch.where((next_owner == 2) & ~stable_b, torch.zeros_like(next_owner), next_owner)

        dead_a = F.max_pool2d(next_state_a[:, 3:4], 3, stride=1, padding=1) <= 0.1
        dead_b = F.max_pool2d(next_state_b[:, 3:4], 3, stride=1, padding=1) <= 0.1

        next_state_a = torch.where(dead_a.expand_as(next_state_a), torch.zeros_like(next_state_a), next_state_a)
        next_state_b = torch.where(dead_b.expand_as(next_state_b), torch.zeros_like(next_state_b), next_state_b)
        next_owner = torch.where(dead_a & (next_owner == 1), torch.zeros_like(next_owner), next_owner)
        next_owner = torch.where(dead_b & (next_owner == 2), torch.zeros_like(next_owner), next_owner)

    return next_state_a, next_state_b, next_owner


def compose_state(state_a, state_b, owner):
    return torch.where(
        (owner == 1).expand_as(state_a),
        state_a,
        torch.where((owner == 2).expand_as(state_b), state_b, torch.zeros_like(state_a)),
    )


def render_surface(state_a, state_b, owner):
    rgba = compose_state(state_a, state_b, owner)[0, :4].detach().cpu().clamp(0.0, 1.0)
    rgb = rgba[:3] * rgba[3:4]
    image = (rgb.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return pygame.surfarray.make_surface(image.swapaxes(0, 1))


def select_target(index, targets):
    index = index % len(targets)
    return index, targets[index]


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--grid-size", type=int, default=48)
    parser.add_argument("--window-size", type=int, default=960)
    parser.add_argument("--fps", type=int, default=30)
    parser.add_argument("--left", type=int, default=1)
    parser.add_argument("--right", type=int, default=2)
    parser.add_argument("--bootstrap-steps", type=int, default=500)
    parser.add_argument("--device", default=pick_device())
    parser.add_argument("--headless-frames", type=int, default=0)
    args = parser.parse_args()

    targets = list_targets()
    if not targets:
        raise SystemExit("no targets found in targets/")

    if args.headless_frames > 0:
        os.environ.setdefault("SDL_VIDEODRIVER", "dummy")

    pygame.init()
    flags = pygame.RESIZABLE
    if args.headless_frames > 0:
        flags |= pygame.HIDDEN
    window = pygame.display.set_mode((args.window_size, args.window_size), flags)
    pygame.display.set_caption("petri clash")
    clock = pygame.time.Clock()

    left_index, left_target = select_target(args.left - 1, targets)
    right_index, right_target = select_target(args.right - 1, targets)
    left_model = ensure_model(left_target, args.device, args.bootstrap_steps)
    right_model = ensure_model(right_target, args.device, args.bootstrap_steps)

    print(f"left  -> {left_target.stem}")
    print(f"right -> {right_target.stem}")

    paused = False
    frames = 0
    state_a, state_b, owner = reset_world(args.grid_size, args.device)

    digit_keys = {
        pygame.K_1: 0,
        pygame.K_2: 1,
        pygame.K_3: 2,
        pygame.K_4: 3,
        pygame.K_5: 4,
        pygame.K_6: 5,
        pygame.K_7: 6,
        pygame.K_8: 7,
        pygame.K_9: 8,
    }

    running = True
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
                elif event.key == pygame.K_SPACE:
                    paused = not paused
                elif event.key == pygame.K_r:
                    state_a, state_b, owner = reset_world(args.grid_size, args.device)
                elif event.key in digit_keys and digit_keys[event.key] < len(targets):
                    target_id = digit_keys[event.key]
                    if event.mod & pygame.KMOD_SHIFT:
                        right_index, right_target = select_target(target_id, targets)
                        right_model = ensure_model(right_target, args.device, args.bootstrap_steps)
                        print(f"right -> {right_target.stem}")
                    else:
                        left_index, left_target = select_target(target_id, targets)
                        left_model = ensure_model(left_target, args.device, args.bootstrap_steps)
                        print(f"left  -> {left_target.stem}")
                    state_a, state_b, owner = reset_world(args.grid_size, args.device)
            elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                width, height = window.get_size()
                gx = int(event.pos[0] * args.grid_size / max(width, 1))
                gy = int(event.pos[1] * args.grid_size / max(height, 1))
                state_a, state_b, owner = crater(
                    state_a,
                    state_b,
                    owner,
                    gx,
                    gy,
                    radius=max(2, args.grid_size // 12),
                )

        if not paused:
            state_a, state_b, owner = clash_step(state_a, state_b, owner, left_model, right_model)

        surface = render_surface(state_a, state_b, owner)
        scaled = pygame.transform.scale(surface, window.get_size())
        window.fill((0, 0, 0))
        window.blit(scaled, (0, 0))
        pygame.display.flip()

        frames += 1
        if args.headless_frames > 0 and frames >= args.headless_frames:
            running = False

        clock.tick(args.fps)

    pygame.quit()


if __name__ == "__main__":
    main()


## the first bad idea: one shared latent grid

my first clash version used one shared 16-channel state grid.

the thought process sounded reasonable at first:

- both models look at the same board
- both propose updates
- ownership decides whose update gets applied per cell

but that sneaks in a fatal assumption.

it assumes model a can safely read hidden channels that were produced by model b, and vice versa.

that is false.

each nca was trained on its own private hidden-state distribution. once i let one model read the other model's latent channels, both were immediately out of distribution. they started producing ugly noise, spreading into weird places, and corrupting their own future decisions.

so the correct fix was not "tune the alpha threshold" or "train more". the correct fix was to stop sharing the latent state.

## the real clash model: separate hidden states, shared territory

the current clash design works like this:

- model a owns its own full 16-channel state grid
- model b owns its own full 16-channel state grid
- the shared thing is only the owner mask
- rendering composes visible rgba from whichever organism currently owns each cell
- conflict resolution happens at the ownership layer, not by mixing hidden states

this is a much safer design because each model always reads activations that came from itself.

that one structural change fixed the nastiest failure mode in the project.

In [ ]:
from clash import ensure_model, reset_world, clash_step, list_targets
import torch

device = 'cpu'
targets = list_targets()
left_model = ensure_model(targets[0], device, 0)
right_model = ensure_model(targets[1], device, 0)
state_a, state_b, owner = reset_world(48, device)

with torch.inference_mode():
    for step in range(20):
        state_a, state_b, owner = clash_step(state_a, state_b, owner, left_model, right_model)

print('owned by a:', int((owner == 1).sum()))
print('owned by b:', int((owner == 2).sum()))
print('unclaimed:', int((owner == 0).sum()))

## why the runtime grid went back to 48x48

at one point i let the clash grid default to 64x64 because it sounded more dramatic.

that was a mistake.

the organisms were trained on 48x48. giving them a larger arena changed the long-horizon dynamics and encouraged late-stage drift into regions they were never trained to stabilize on.

so the default went back to 48x48.

that is one of those boring fixes that matters more than flashy ones. a lot of "ai system looks weird at runtime" bugs are just train-test mismatch.

## why there is support pruning in the clash loop

after the separate-state fix, there was still another ugly behavior: tiny sparks and filaments could survive in odd places, then later become seeds for junk growth.

so i added a simple structural rule:

- if a claimed cell does not have enough same-owner support in its local 3x3 neighborhood, kill it

this is not a paper-derived nca rule. it is a game-side stabilizer.

i added it because petri clash is not trying to be a pure research reproduction anymore. it is trying to behave well as an interactive system.

that tradeoff matters. once something becomes a toy you can poke with the mouse, stability starts to matter as much as theoretical purity.

In [ ]:
import torch
from clash import ensure_model, reset_world, clash_step, list_targets

def footprint(size=48, steps=200):
    device = 'cpu'
    targets = list_targets()
    left = ensure_model(targets[0], device, 0)
    right = ensure_model(targets[1], device, 0)
    state_a, state_b, owner = reset_world(size, device)
    with torch.inference_mode():
        rows = []
        for step in range(1, steps + 1):
            state_a, state_b, owner = clash_step(state_a, state_b, owner, left, right)
            if step % 40 == 0:
                a = int((owner == 1).sum())
                b = int((owner == 2).sum())
                rows.append((step, a, b, a + b))
    return rows

for row in footprint():
    print(row)

## the pygame loop

i kept the interface intentionally bare.

there is no hud, no graphs, no side panel, no settings menu, no tutorial screen.

that was not laziness. it was a taste decision.

the whole point of the project is to stare at the board itself. everything else would just distract from the organisms.

so the pygame loop only really does five jobs:

- open a window
- load the requested left and right targets
- step the world when not paused
- translate keyboard and mouse input into world changes
- render the composed rgba image scaled up to window size

that is exactly the amount of ui the idea needs.

## controls and why they are so minimal

these are the controls:

- `space` pause or resume
- `r` reset the board with new seed locations
- `1` through `9` change the left organism
- `shift+1` through `shift+9` change the right organism
- left click punches a crater
- `esc` quits

this keeps the interaction loop tight.

i did not want any action in the project to require leaving the visual field. that is why even target switching lives on the number keys.

## the target pack

the target icons are deliberately small and simple.

this was not because the system could never learn more detailed images. it was because these shapes are enough to expose the dynamics clearly:

- heart shows whether rounded mass can hold together
- star shows whether sharp protrusions survive
- the others give variety without requiring a huge image dataset

for a project like this, the first job of the targets is to reveal behavior, not to impress with complexity.

## the sequence of bugs and fixes

this is the part i especially wanted to document, because it explains why the code looks the way it looks now.

### phase 1: get a real nca running

i built `nca.py` and `train.py`, verified that a seed could grow, and made sure checkpoints saved correctly.

### phase 2: get a playable clash loop running

i built a simple shared-grid clash loop in pygame. it technically ran, but the results were ugly.

### phase 3: fix the fake weights problem

i noticed that tiny smoke-test checkpoints were being loaded like they were real models. i added `train_steps` to checkpoints and made the loader refresh weak weights.

### phase 4: train the first real models

i trained the heart and star to 3000 steps and pushed those weights.

### phase 5: find the real clash bug

i realized the shared latent state was poisoning both models, because each one was reading hidden channels it was never trained to interpret.

### phase 6: separate the hidden states

i gave each organism its own private 16-channel state and shared only ownership plus rendering.

### phase 7: reduce late drift

i put the runtime grid back to 48x48 and added neighborhood-support pruning so tiny sparks would not become future garbage colonies.

that history is the reason the final code is not the shortest possible version. it is the version that survived contact with actual behavior.

## what i would improve next

if i kept pushing this project, these are the next things i would try.

### 1. train with clash-aware perturbations

right now the models are trained for solo growth plus crater damage. they are not explicitly trained against another live organism.

one next step would be to inject adversarial occupancy masks or partial territorial blocking during training.

### 2. separate visible state and combat confidence

right now alpha is doing two jobs:

- visible opacity
- conflict confidence

that is convenient, but maybe too entangled. a dedicated combat channel might make frontier decisions cleaner.

### 3. use living-neighbor constraints during training too

support pruning currently exists in the clash runtime. maybe a similar inductive bias should exist during training so the models learn less spark-prone dynamics naturally.

### 4. add reproducible evaluation scripts

i would like a small benchmark that measures:

- target reconstruction loss over long rollouts
- recovery after crater damage
- occupied area drift after 200 or 400 steps
- territorial stability in a two-model clash

that would make iteration much less subjective.

## how to run the project now

from the repo root:

```bash
conda activate petri-clash
python clash.py --bootstrap-steps 0
```

for training a target explicitly:

```bash
python train.py --target targets/01_heart.png --steps 3000
```

for testing the same target on both sides:

```bash
python clash.py --left 1 --right 1 --bootstrap-steps 0
```

that last one is useful when you want to see whether a weird behavior is coming from the clash logic or from one weak model.

## final thought

what i like about this project is that it only looks simple from far away.

"two little growing things fight on a board" sounds tiny. but the moment you actually implement it, you run straight into the hard parts of dynamical systems:

- hidden state distributions matter
- train-time and test-time geometry have to match
- local mistakes can become future seeds
- a model that works alone can fail badly in interaction

so the final implementation is really a sequence of guardrails around a fragile but beautiful core idea.

that is the real story of how this repo got built.